# ForensiX: video & image detector benchmark on SDFVD

Runtime → Change runtime type → **T4 GPU** (or better).

SDFVD: 53 real (Pexels) + 53 face-swapped videos, CC BY 4.0, 182 MB — https://data.mendeley.com/datasets/bcmkfgct2s/1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!if [ ! -d ForensiX ]; then git clone https://github.com/anas-rz/ForensiX.git; fi
%cd ForensiX
!git pull -q
!pip install -q -e .

In [ ]:
# Download SDFVD and check its hash
!curl -sL -o /content/SDFVD.zip "https://data.mendeley.com/public-files/datasets/bcmkfgct2s/files/a9ef71a3-b625-4682-a21c-f0579e85cdd4/file_downloaded"
!echo "734a101fdfd89d5b20ac69f510d6a7cf3b588a145c1993ba2488322ae50d95a5  /content/SDFVD.zip" | sha256sum -c
!unzip -q -o /content/SDFVD.zip -d /content/sdfvd && ls /content/sdfvd/SDFVD

Optional: mount Drive so results survive a disconnect. The script resumes from `--out`, so rerunning the next cell skips finished models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/forensix_results'
!mkdir -p {OUT}

In [ ]:
OUT = globals().get('OUT', '/content/results')
!python scripts/eval_video_folder.py --device cuda \
    --real /content/sdfvd/SDFVD/videos_real --fake /content/sdfvd/SDFVD/videos_fake --dataset SDFVD \
    --models univfd rine rine-ldm commfor commfor-224 safe clipdet gend gend-pe dfdet-clip dfdc-wm dfdc-selim \
    --out {OUT}/sdfvd.json --scores-out {OUT}/sdfvd_scores.csv

In [ ]:
import json
r = json.load(open(f'{OUT}/sdfvd.json'))
print('| Model | Family | AUC | AP | EER | Bal. acc | Real acc | Fake acc | Failed | Time (s) |')
print('|---|---|---|---|---|---|---|---|---|---|')
for name, m in sorted(r['models'].items(), key=lambda kv: -kv[1]['auc']):
    print(f"| {name} | {m['family']} | {m['auc']:.3f} | {m['ap']:.3f} | {m['eer']:.3f} | {m['balanced_acc']:.3f} | {m['real_acc']:.3f} | {m['fake_acc']:.3f} | {m['failed']} | {m['seconds']:.0f} |")